# Base Model

## Imports

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

## Loading Data

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = datasets.MNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

test_dataset = datasets.MNIST(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

TypeError: MNIST.__init__() missing 1 required positional argument: 'root'

In [3]:
print("Training samples:", len(train_dataset))
print("Test samples:", len(test_dataset))

image, label = train_dataset[0]
print("\nImage shape:", image.shape)
print("Label:", label)

Training samples: 60000
Test samples: 10000

Image shape: torch.Size([1, 28, 28])
Label: 5


## Config Settings

In [4]:
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)
print("Torch version:", torch.__version__)

CUDA available: True
CUDA version: 12.1
Torch version: 2.5.1+cu121


In [5]:
# device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(device)

cuda


## Model Code

In [6]:
class ShallowNN(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.fc1 = nn.Linear(28 * 28, hidden_size)
        self.fc2 = nn.Linear(hidden_size, 10)

    def forward(self, x):
        x = x.view(x.size(0), -1)      # flatten
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)
        return x

In [10]:
class Trainer:
    def __init__(self, hidden_size, learning_rate, epochs, batch_size):
        self.hidden_size = hidden_size
        self.learning_rate = learning_rate
        self.epochs = epochs
        self.batch_size = batch_size

        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        self.model = ShallowNN(hidden_size).to(self.device)
        self.optimizer = optim.Adam(self.model.parameters(), lr=learning_rate)
        self.criterion = nn.CrossEntropyLoss()

        self.init_data_loaders()

    def init_data_loaders(self):
        self.train_loader = DataLoader(
            train_dataset,
            batch_size=self.batch_size,
            shuffle=True
        )

        self.test_loader = DataLoader(
            test_dataset,
            batch_size=1000,
            shuffle=False
        )

    def train(self, debug=False):
        for epoch in range(1, self.epochs + 1):
            self.model.train()
            for data, target in self.train_loader:
                data, target = data.to(self.device), target.to(self.device)

                self.optimizer.zero_grad()
                output = self.model(data)
                loss = self.criterion(output, target)
                loss.backward()
                self.optimizer.step()

            acc = self.evaluate()

            if debug:
                print(f"Epoch {epoch}/{self.epochs} | Test Accuracy: {acc:.4f}")

    def evaluate(self):
        self.model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for data, target in self.test_loader:
                data, target = data.to(self.device), target.to(self.device)
                output = self.model(data)
                pred = output.argmax(dim=1)
                correct += (pred == target).sum().item()
                total += target.size(0)
        return correct / total

## Testing

In [13]:
trainer = Trainer(
    hidden_size=64,
    learning_rate=1e-3,
    epochs=10,
    batch_size=512
)

trainer.train(debug=True)

Epoch 1/10 | Test Accuracy: 0.9195
Epoch 2/10 | Test Accuracy: 0.9395
Epoch 3/10 | Test Accuracy: 0.9517
Epoch 4/10 | Test Accuracy: 0.9570
Epoch 5/10 | Test Accuracy: 0.9588
Epoch 6/10 | Test Accuracy: 0.9641
Epoch 7/10 | Test Accuracy: 0.9662
Epoch 8/10 | Test Accuracy: 0.9685
Epoch 9/10 | Test Accuracy: 0.9683
Epoch 10/10 | Test Accuracy: 0.9706


In [15]:
print(f"Test Accuracy: {trainer.evaluate()}")

Test Accuracy: 0.9706
